In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

windowed_df = pd.read_csv('../output/windown_sliced_data/windowed_df_10d.csv', parse_dates=['date', 'window_start', 'window_end', 'test_date'])
windowed_test = windowed_df.query('is_test_day == True').copy()
windowed_test

,date,patient_id,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen,window_start,window_end,test_date,is_test_day
10,2019-05-08,16f4b,0.621309,0.0,0.750000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,True
21,2019-05-13,16f4b,0.623305,0.0,1.500000,0.0,0.0,2019-05-03,2019-05-12,2019-05-13,True
32,2019-05-05,1fbe4,0.681941,1.0,0.800000,0.0,0.0,2019-04-25,2019-05-04,2019-05-05,True
43,2019-05-06,1fbe4,0.668119,0.0,0.000000,0.0,0.0,2019-04-26,2019-05-05,2019-05-06,True
54,2019-05-07,1fbe4,0.681583,1.0,1.250000,0.0,1.0,2019-04-27,2019-05-06,2019-05-07,True
...,...,...,...,...,...,...,...,...,...,...,...
5367,2019-06-08,ec812,0.649826,0.0,1.285714,0.0,0.0,2019-05-29,2019-06-07,2019-06-08,True
5378,2019-06-22,ec812,0.623560,0.0,0.666667,0.0,0.0,2019-06-12,2019-06-21,2019-06-22,True
5389,2019-06-23,ec812,0.708682,0.0,0.875000,0.0,0.0,2019-06-13,2019-06-22,2019-06-23,True
5400,2019-06-24,ec812,0.656330,0.0,0.888889,0.0,0.0,2019-06-14,2019-06-23,2019-06-24,True


In [2]:
import pandas as pd

df_label = pd.read_csv("../output/Anomaly_delirium_Revised/LSTM_anomaly_labels.csv")
df_label

,patient_id,window_start,test_date,baseline_error,test_error,anomaly_score,threshold,anomaly_label
0,16f4b,2019-04-28,2019-05-08,0.145082,0.142482,-0.002600,-0.003909,-1
1,16f4b,2019-05-03,2019-05-13,0.171185,0.159862,-0.011322,-0.003909,1
2,1fbe4,2019-04-25,2019-05-05,0.197504,0.189899,-0.007605,0.020904,1
3,1fbe4,2019-04-26,2019-05-06,0.189899,0.185698,-0.004201,0.020904,1
4,1fbe4,2019-04-27,2019-05-07,0.185698,0.200494,0.014796,0.020904,1
...,...,...,...,...,...,...,...,...
487,ec812,2019-05-29,2019-06-08,0.168403,0.193873,0.025471,0.009404,-1
488,ec812,2019-06-12,2019-06-22,0.046762,0.057492,0.010729,0.009404,-1
489,ec812,2019-06-13,2019-06-23,0.057490,0.063119,0.005629,0.009404,1
490,ec812,2019-06-14,2019-06-24,0.061634,0.064390,0.002756,0.009404,1


In [3]:
import pandas as pd

# shared keys
keys = ["patient_id", "test_date"]

# make sure the merge key has the same dtype in both dataframes
df_label["test_date"] = pd.to_datetime(df_label["test_date"])
windowed_test["test_date"] = pd.to_datetime(windowed_test["test_date"])

# only keep selected columns from windowed_test
B_sub = windowed_test[
    keys + [
        "entropy_rate",
        "early_warning_score",
        "sleep_quality_score",
        "agitation_counts",
        "uti_happen"
    ]
    ]

# merge
LSTM_visual_df = df_label.merge(B_sub, on=keys, how="left")

LSTM_visual_df

,patient_id,window_start,test_date,baseline_error,test_error,anomaly_score,threshold,anomaly_label,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen
0,16f4b,2019-04-28,2019-05-08,0.145082,0.142482,-0.002600,-0.003909,-1,0.621309,0.0,0.750000,0.0,0.0
1,16f4b,2019-05-03,2019-05-13,0.171185,0.159862,-0.011322,-0.003909,1,0.623305,0.0,1.500000,0.0,0.0
2,1fbe4,2019-04-25,2019-05-05,0.197504,0.189899,-0.007605,0.020904,1,0.681941,1.0,0.800000,0.0,0.0
3,1fbe4,2019-04-26,2019-05-06,0.189899,0.185698,-0.004201,0.020904,1,0.668119,0.0,0.000000,0.0,0.0
4,1fbe4,2019-04-27,2019-05-07,0.185698,0.200494,0.014796,0.020904,1,0.681583,1.0,1.250000,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
487,ec812,2019-05-29,2019-06-08,0.168403,0.193873,0.025471,0.009404,-1,0.649826,0.0,1.285714,0.0,0.0
488,ec812,2019-06-12,2019-06-22,0.046762,0.057492,0.010729,0.009404,-1,0.623560,0.0,0.666667,0.0,0.0
489,ec812,2019-06-13,2019-06-23,0.057490,0.063119,0.005629,0.009404,1,0.708682,0.0,0.875000,0.0,0.0
490,ec812,2019-06-14,2019-06-24,0.061634,0.064390,0.002756,0.009404,1,0.656330,0.0,0.888889,0.0,0.0


# SHADED VERSION (IN BLUE COLOR TO BE DIFFERENT FROM IF algorithm)


In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

for id_select in LSTM_visual_df['patient_id'].unique():
    df_person = LSTM_visual_df[LSTM_visual_df['patient_id'] == id_select].copy()

    plt.figure(figsize=(12, 10))

    anomaly_dates_dt = pd.to_datetime(
        df_person[df_person.anomaly_label == -1].test_date
    ).sort_values()
    anomaly_dates = anomaly_dates_dt.dt.strftime('%m-%d').astype(str).values

    # ── identify consecutive clusters based on calendar dates ────────────────
    clusters = []
    if len(anomaly_dates_dt) > 0:
        cluster_start = anomaly_dates_dt.iloc[0]
        cluster_end = anomaly_dates_dt.iloc[0]

        for date in anomaly_dates_dt.iloc[1:]:
            if (date - cluster_end).days == 1:
                cluster_end = date
            else:
                if (cluster_end - cluster_start).days >= 1:  # at least 2 consecutive days
                    clusters.append((
                        cluster_start.strftime('%m-%d'),
                        cluster_end.strftime('%m-%d')
                    ))
                cluster_start = date
                cluster_end = date
        # check last cluster
        if (cluster_end - cluster_start).days >= 1:
            clusters.append((
                cluster_start.strftime('%m-%d'),
                cluster_end.strftime('%m-%d')
            ))

    data_plot = df_person[['test_date', 'entropy_rate', 'early_warning_score',
                           'sleep_quality_score', 'agitation_counts', 'uti_happen']].copy()
    data_plot.rename(columns={
        "entropy_rate": "entropy rate",
        "early_warning_score": "early warning score",
        "sleep_quality_score": "sleep quality score",
        "agitation_counts": "agitation counts",
        "uti_happen": "uti counts"
    }, inplace=True)
    data_plot['test_date'] = pd.to_datetime(data_plot['test_date']).dt.strftime('%m-%d').astype(str)
    data_plot.set_index('test_date', inplace=True)
    x_labels = data_plot.index.tolist()

    for i, column in enumerate(data_plot.columns):
        ax = plt.subplot(5, 1, i + 1)
        ax.plot(x_labels, data_plot[column], label=column)
        ax.scatter(anomaly_dates, data_plot[column][anomaly_dates],
                   color='red', label='anomaly', zorder=5)

        # ── shade consecutive clusters ────────────────────────────────────────
        for cluster_start, cluster_end in clusters:
            if cluster_start in x_labels and cluster_end in x_labels:
                x_start = x_labels.index(cluster_start)
                x_end = x_labels.index(cluster_end)
                ax.axvspan(x_start - 0.5, x_end + 0.5,
                           alpha=0.2, color='blue', zorder=0)

        ax.set_title(f'{column}')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xticks(range(len(x_labels)))
        ax.set_xticklabels(x_labels, rotation=90)
        ax.set_ylim(data_plot[column].min() - 1, data_plot[column].max() + 1)

    plt.tight_layout()
    plt.savefig('../output/Anomaly_delirium_Revised/LSTM_visual_shaded/anomaly_ForestIsolation_' + id_select + '.png', dpi=300)
    plt.close()